In [25]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn

In [9]:
class LinearRegression:
  def __init__(self, n_iter, lr = 0.01):
    self.n_iter = n_iter
    self.lr = lr


  def train(self, X_train, y_train, type = "SGD", batch = 32, c = 0):
    n, m = X_train.shape
    self.weights = np.zeros((m,1))
    self.bias = 0
    y_train = y_train.reshape(-1, 1)

    if type == "SGD":
      for _ in range(self.n_iter):
        for i in range(n):
          y_hat = np.dot(X_train[i], self.weights) + self.bias
          error = y_hat - y_train[i]
          # for j in range(len(self.weights)):
          #   self.weights[j] -= (self.lr * error) * X_train[i][j]
          self.weights -= self.lr * (error * X_train[i].reshape(-1,1) + c * self.weights)
          self.bias -= self.lr * error

    elif type == 'BGD':
      for _ in range(self.n_iter):
        dw = np.zeros_like(self.weights)
        db = 0
        # for i in range(n):
        #   y_hat = np.dot(X_train[i], self.weights) + self.bias
        #   error = y_hat - y_train[i]
        #   dw += (error * X_train[i].reshape(-1,1))
        #   db += error
        y_hat = X_train @ self.weights + self.bias
        error = y_hat - y_train
        dw = X_train.T @ error
        db = np.sum(error)
        self.weights -= self.lr * ((dw + c * self.weights) / n)
        self.bias -= self.lr * db / n

    else:
       for _ in range(self.n_iter):
        for i in range(0, n, batch):
          end = min(i + batch, n)
          X_batch = X_train[i:end]
          y_batch = y_train[i:end]

          # for j in range(i, end):
          #   batch_size = end - i
          #   y_hat = np.dot(X_train[j], self.weights) + self.bias
          #   error = y_hat - y_train[j]
          #   dw += (error * X_train[j].reshape(-1,1))
          #   db += error
          y_hat = X_batch @ self.weights + self.bias
          error = y_hat - y_batch
          dw = X_batch.T @ error
          db = np.sum(error)

          self.weights -= self.lr * ((dw + c * self.weights) / (end - i))
          self.bias -= self.lr * db / (end - i)

  def predict(self, X_test):
    y_pred = np.dot(X_test, self.weights) + self.bias
    return y_pred

In [10]:
# test data
np.random.seed(42)
X = np.random.rand(100, 1) * 10
y = 4 * X + 3 + np.random.randn(100, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
lr = LinearRegression(100)

In [15]:
lr.train(X_train, y_train, type="BGD")

In [16]:
print(lr.weights, lr.bias)

[[4.25179359]] 1.26443577445699


## Using Pytorch

In [18]:
class LinearRegression_torch:
  def __init__(self, n_iter, lr = 0.01):
    self.n_iter = n_iter
    self.lr = lr


  def train(self, X_train, y_train):
    X_train = torch.tensor(X_train, dtype = torch.float32)
    y_train = torch.tensor(y_train, dtype = torch.float32).reshape(-1,1)
    
    n, m = X_train.shape
    self.weights = torch.zeros((m,1), requires_grad=True)
    self.bias = torch.tensor(0.0, requires_grad=True)

    for _ in range(self.n_iter):
      y_hat = X_train @ self.weights + self.bias
      loss = ((y_hat - y_train) ** 2).mean()
     
      loss.backward()

      with torch.no_grad():
        self.weights -= self.lr * self.weights.grad
        self.bias -= self.lr * self.bias

      self.weights.grad.zero_()
      self.bias.grad.zero_()

  def predict(self, X_test):
    X_test = torch.tensor(X_test, dtype = torch.float32)
    y_pred = X_test @ self.weights + self.bias
    return y_pred

In [22]:
lr_torch = LinearRegression_torch(1000)

In [23]:
lr_torch.train(X_train, y_train)

In [24]:
print(lr.weights, lr.bias)

[[4.25179359]] 1.26443577445699


## Using torch nn module

In [32]:
class LR(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    self.linear = nn.Linear(input_dim, 1)

  def forward(self, x):
    x = torch.tensor(x, dtype = torch.float32)
    return self.linear(x)

In [37]:
model = LR(X_train.shape[1])
y_train = torch.tensor(y_train, dtype = torch.float32)
for _ in range(1000):
  y_hat = model(X_train)
  loss = ((y_hat - y_train ) ** 2 ).mean()

  loss.backward()

  with torch.no_grad():
    model.linear.weight -= 0.01 * model.linear.weight.grad
    model.linear.bias -= 0.01 * model.linear.bias.grad

  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()

/var/folders/vs/57gy8j1n2077056vd0whtvkw0000gn/T/ipykernel_40773/2944209475.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype = torch.float32)


In [38]:
print(model.linear.weight, model.linear.bias)

Parameter containing:
tensor([[3.9613]], requires_grad=True) Parameter containing:
tensor([3.1338], requires_grad=True)
